In [ ]:
"""
AlphaPortfolio v4 — T+5 label + CVaR + turnover penalty
Inference notebook for BigAlpha 2026
"""
import os
import numpy as np
import pandas as pd
import dai
import torch
import structlog

from transformer_train import (
    MODEL_PATH, BATCH, ENSEMBLE_N, StockTransformer, build_dataset, pool, load_model,
)

logger = structlog.get_logger()


def main(datasources, start_date, end_date):
    """加载T+5模型, ensemble推理打分."""
    table = datasources["bar5m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"未找到模型文件 {MODEL_PATH}")

    ckpt = load_model(MODEL_PATH, map_location=device)
    stats = (np.asarray(ckpt["mean"], np.float32), np.asarray(ckpt["std"], np.float32))
    model = StockTransformer(**ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    logger.info("loaded model", path=MODEL_PATH, params=sum(p.numel() for p in model.parameters()))

    logger.info("building test set", start=str(start_date), end=str(end_date))
    Xte, _, idx_df, _ = build_dataset(table, start_date, end_date, "infer",
                                      pool(start_date, end_date), stats)

    # Ensemble inference
    logger.info(f"ensemble inference (N={ENSEMBLE_N})")
    preds = []
    Xte_t = torch.from_numpy(Xte)
    bs = BATCH // ENSEMBLE_N
    with torch.no_grad():
        for i in range(0, len(idx_df), bs):
            xb = Xte_t[i:i+bs].to(device)
            ensemble_pred = model.predict_with_ensemble(xb, n=ENSEMBLE_N)
            preds.append(ensemble_pred.cpu().numpy())
    idx_df["score"] = np.concatenate(preds).astype(np.float64)

    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    result = (pd.merge(idx_df, stk, on=["date","instrument"], how="inner")
                .replace([np.inf,-np.inf], np.nan).dropna(subset=["score"])
                .drop_duplicates(["date","instrument"])[["date","instrument","score"]]
                .reset_index(drop=True))
    logger.info("done", rows=len(result))
    return result
